In [ ]:
!pip install torch torchvision torchaudio transformers

!pip install kobert-transformers sentencepiece

!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

  Cloning https://github.com/SKTBrain/KoBERT.git to /tmp/pip-install-wy6sm4d7/kobert-tokenizer_3182dbe9abee42ae9cf3621f0e879437
  Running command git clone --filter=blob:none --quiet https://github.com/SKTBrain/KoBERT.git /tmp/pip-install-wy6sm4d7/kobert-tokenizer_3182dbe9abee42ae9cf3621f0e879437
  Resolved https://github.com/SKTBrain/KoBERT.git to commit fcd729f2f4b37858f333597c0782388ada51eb5f
  Preparing metadata (setup.py) ... done
  Created wheel for kobert_tokenizer: filename=kobert_tokenizer-0.1-py3-none-any.whl size=4633 sha256=4d2fbd132d69fe204fdabab8fba8dec02573ce421d564cb6e0b5aa356ff584a1
  Stored in directory: /tmp/pip-ephem-wheel-cache-h4qe_wx6/wheels/ec/ae/fb/30f74ad83a90c3f950522723b8d918d197fa77d6bd7df7b028
Successfully built kobert_tokenizer


In [ ]:
import os, random, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW                     
from transformers import AutoTokenizer, AutoModel  
from transformers import get_cosine_schedule_with_warmup, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from google.colab import drive

from kobert_tokenizer import KoBERTTokenizer

In [ ]:
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


In [ ]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)

In [ ]:
# 학습
lr = 3e-5
num_epochs = 12           
batch_size = 8
accum_steps = 2
warmup_ratio = 0.02
grad_clip_norm = 0.8
patience = 8

# 데이터
max_len = 80

# 모델/정규화
dr_rate = 0.3            
llrd_lr_decay = 0.9
wd_base = 0.01
wd_decay = 0.9
no_decay_wd = 0.0
head_lr_mult = 3.0        

cb_beta = 0.999
focal_gamma = 1.5


In [ ]:
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/lawtalk_data.csv'
df = pd.read_csv(file_path, encoding='utf-8')

Mounted at /content/drive


In [ ]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['interest'])
num_classes = len(label_encoder.classes_)
NUM_LABELS = num_classes

In [ ]:
#계층적 샘플링

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
def tokenize_title(title):
    encoding = tokenizer.encode_plus(
        title,
        add_special_tokens=True,
        max_length=64,
        truncation=True
    )
    return tokenizer.convert_ids_to_tokens(encoding['input_ids'])

df['decoded_tokens'] = df['title'].apply(tokenize_title)
df_tok = df[['title', 'decoded_tokens']].head(10)
print(df_tok.to_markdown())

|    | title                                            | decoded_tokens                                                                                                   |
|---:|:-------------------------------------------------|:-----------------------------------------------------------------------------------------------------------------|
|  0 | 이혼 후 재산분할 미지급 시 대처 방안             | ['[CLS]', '▁이혼', '▁후', '▁재산', '분', '할', '▁미', '지', '급', '▁시', '▁대처', '▁방안', '[SEP]']              |
|  1 | 직장 내 성추행 사건에 대한 고소 준비             | ['[CLS]', '▁직장', '▁내', '▁성추행', '▁사건', '에', '▁대한', '▁고소', '▁준비', '[SEP]']                          |
|  2 | 전세권 설정과 관련된 질문                        | ['[CLS]', '▁전세', '권', '▁설정', '과', '▁관련된', '▁질문', '[SEP]']                                             |
|  3 | 폭행사건 피해자, 민사소송 가능성에 대한 상담     | ['[CLS]', '▁폭행', '사건', '▁피해자', ',', '▁민', '사', '소송', '▁가능성', '에', '▁대한', '▁상담', '[SEP]']      |
|  4 | 스토킹 및 협박 고소에 대한 우려와 대처 방안      | ['[CLS]', '▁스', '토', '킹', '▁및', '▁협박', '▁고

In [ ]:
class BERTDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.titles = df['title'].astype(str).tolist()
        self.labels = df['label'].astype(int).tolist()
        self.tok = tokenizer; self.max_len = max_len

    def __len__(self): return len(self.titles)

    def __getitem__(self, i):
        enc = self.tok(
            self.titles[i],
            padding="max_length", truncation=True, max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}

        if "token_type_ids" not in item:
            item["token_type_ids"] = torch.zeros_like(item["input_ids"])
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

train_dataset = BERTDataset(train_df, tokenizer, max_len)
test_dataset  = BERTDataset(test_df, tokenizer, max_len)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, drop_last=False)

In [ ]:
class KoBertClassifier(nn.Module):
    def __init__(self, num_classes, dr_rate=0.3, bert=None):
        super().__init__()
        self.bert = AutoModel.from_pretrained("skt/kobert-base-v1")
        hidden = self.bert.config.hidden_size
        self.layernorm = nn.LayerNorm(hidden)
        self.classifier = nn.Sequential(
            nn.Dropout(p=dr_rate),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(p=dr_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, token_type_ids, attention_mask):
        out = self.bert(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state             
        mask = attention_mask.unsqueeze(-1).float()       
        pooled = (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1.0)  
        pooled = self.layernorm(pooled)
        return self.classifier(pooled)

model = KoBertClassifier(num_classes=NUM_LABELS, dr_rate=dr_rate).to(device)


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

In [ ]:
def build_param_groups_llrd_safe(model, base_lr=3e-5, lr_decay=0.9,
                                 wd_base=0.01, wd_decay=1.0, no_decay_wd=0.0,
                                 head_lr_mult=3.0):
    no_decay = ("bias", "LayerNorm.weight")
    groups = []
    def add(params, lr, wd):
        if params: groups.append({"params": params, "lr": float(max(lr,1e-8)), "weight_decay": float(max(wd,0.0))})

    emb_d, emb_nd = [], []
    for n,p in model.named_parameters():
        if n.startswith("bert.embeddings."):
            (emb_nd if any(nd in n for nd in no_decay) else emb_d).append(p)
    add(emb_d,  base_lr*(lr_decay**12), wd_base*(wd_decay**12))
    add(emb_nd, base_lr*(lr_decay**12), no_decay_wd)

    for depth in range(12):
        prefix = f"bert.encoder.layer.{11-depth}."
        lr_i = base_lr*(lr_decay**depth); wd_i = wd_base*(wd_decay**depth)
        d, nd = [], []
        for n,p in model.named_parameters():
            if n.startswith(prefix):
                (nd if any(x in n for x in no_decay) else d).append(p)
        add(d, lr_i, wd_i); add(nd, lr_i, no_decay_wd)

    pool_d, pool_nd = [], []
    for n,p in model.named_parameters():
        if n.startswith("bert.pooler."):
            (pool_nd if any(x in n for x in no_decay) else pool_d).append(p)
    add(pool_d,  base_lr, wd_base); add(pool_nd, base_lr, no_decay_wd)

    head_d, head_nd = [], []
    for n,p in model.named_parameters():
        if not n.startswith("bert."):
            (head_nd if any(x in n for x in no_decay) else head_d).append(p)
    add(head_d,  base_lr*head_lr_mult, wd_base)
    add(head_nd, base_lr*head_lr_mult, no_decay_wd)

    covered = {id(p) for g in groups for p in g["params"]}
    leftovers = [p for _,p in model.named_parameters() if id(p) not in covered]
    add(leftovers, base_lr, wd_base)
    return groups

param_groups = build_param_groups_llrd_safe(model,
    base_lr=lr, lr_decay=llrd_lr_decay, wd_base=wd_base, wd_decay=wd_decay,
    no_decay_wd=no_decay_wd, head_lr_mult=head_lr_mult)
optimizer = AdamW(param_groups)

total_steps  = (len(train_dataloader) * num_epochs) // accum_steps
warmup_steps = int(total_steps * warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)


In [ ]:
import torch.nn.functional as F

def get_class_balanced_weights(labels_np, num_classes, beta=0.999):
    import numpy as np
    counts = np.bincount(labels_np, minlength=num_classes).astype(float)
    eff = 1.0 - np.power(beta, counts)
    w = (1.0 - beta) / np.clip(eff, 1e-12, None)
    w = w / w.sum() * num_classes
    return torch.tensor(w, dtype=torch.float)

class CB_FocalLoss(nn.Module):
    def __init__(self, class_weights: torch.Tensor, gamma: float = 1.5):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.gamma = gamma
    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=-1)
        p = logp.exp()
        ce = F.nll_loss(logp, targets, reduction="none")
        pt = p.gather(1, targets.unsqueeze(1)).squeeze(1).clamp_(1e-8, 1-1e-8)
        w = self.class_weights[targets]
        return (w * ((1-pt)**self.gamma) * ce).mean()

cb_weights = get_class_balanced_weights(train_df['label'].values, NUM_LABELS, beta=cb_beta).to(device)
loss_fn = CB_FocalLoss(class_weights=cb_weights, gamma=focal_gamma)


In [ ]:
def accuracy_from_logits(logits, labels):
    preds = logits.argmax(-1)
    return (preds == labels).float().mean().item()

def train(model, loader, optimizer, scheduler, loss_fn, device, accum_steps=1, grad_clip_norm=None):
    model.train()
    running_loss, running_acc, n = 0.0, 0.0, 0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, total=len(loader), desc="Train", leave=False)
    for step, batch in enumerate(pbar, start=1):
        input_ids = batch["input_ids"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, token_type_ids, attention_mask)
        loss = loss_fn(logits, labels) / accum_steps
        loss.backward()

        if grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

        if step % accum_steps == 0:
            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)

        bs = labels.size(0)
        running_loss += loss.item() * bs * accum_steps
        running_acc  += (logits.argmax(-1) == labels).float().sum().item()
        n += bs

        cur_lr = scheduler.get_last_lr()[-1] if hasattr(scheduler, "get_last_lr") else optimizer.param_groups[0]["lr"]
        pbar.set_postfix(loss=f"{running_loss/max(1,n):.4f}", lr=f"{cur_lr:.2e}")

    return running_loss / max(1,n), running_acc / max(1,n)

    return running_loss / max(1,n), running_acc / max(1,n)

@torch.inference_mode()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    running_loss, running_acc, n = 0.0, 0.0, 0

    pbar = tqdm(loader, total=len(loader), desc="Val", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, token_type_ids, attention_mask)
        loss = loss_fn(logits, labels)

        bs = labels.size(0)
        running_loss += loss.item() * bs
        running_acc  += (logits.argmax(-1) == labels).float().sum().item()
        n += bs

        pbar.set_postfix(loss=f"{running_loss/max(1,n):.4f}")

    return running_loss / max(1,n), running_acc / max(1,n)



In [ ]:
class EarlyStopping:
    def __init__(self, patience=8, path="best_kobert_model.pt", verbose=True):
        self.patience = patience; self.counter = 0; self.best = None
        self.early_stop = False; self.path = path; self.verbose = verbose
    def __call__(self, val_loss, model):
        if self.best is None or val_loss < self.best:
            self.best = val_loss; self.counter = 0
            torch.save(model.state_dict(), self.path)
            if self.verbose: print(f"Validation loss decreased. Saving model to {self.path}")
        else:
            self.counter += 1
            if self.verbose: print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience: self.early_stop = True


In [ ]:
print(f"\n--KoBERT 학습 시작 (epochs={num_epochs}) ---")
early_stopping = EarlyStopping(patience=patience, path="best_kobert_model.pt", verbose=True)

best_val_acc = 0.0
for epoch in range(num_epochs):
    tr_loss, tr_acc = train(model, train_dataloader, optimizer, scheduler, loss_fn, device,
                            accum_steps=accum_steps, grad_clip_norm=grad_clip_norm)
    val_loss, val_acc = evaluate(model, test_dataloader, loss_fn, device)
    print(f"Epoch {epoch+1:02d} | Train {tr_loss:.4f}/{tr_acc:.4f} | Val {val_loss:.4f}/{val_acc:.4f}")

    early_stopping(val_loss, model)
    best_val_acc = max(best_val_acc, val_acc)
    if early_stopping.early_stop:
        print("Early stopping triggered."); break

model.load_state_dict(torch.load("best_kobert_model.pt", map_location=device))
print(f"Best Val Acc: {best_val_acc:.4f}")



--KoBERT 학습 시작 (epochs=12) ---


Train:   0%|          | 0/5488 [00:00<?, ?it/s]

Val:   0%|          | 0/1372 [00:00<?, ?it/s]

Epoch 01 | Train 0.6707/0.4799 | Val 0.5815/0.5134
Validation loss decreased. Saving model to best_kobert_model.pt


Train:   0%|          | 0/5488 [00:00<?, ?it/s]

...

Val:   0%|          | 0/1372 [00:00<?, ?it/s]

Epoch 08 | Train 0.2951/0.6602 | Val 0.6783/0.5052
EarlyStopping counter: 7/8


Train:   0%|          | 0/5488 [00:00<?, ?it/s]

Val:   0%|          | 0/1372 [00:00<?, ?it/s]

Epoch 09 | Train 0.2637/0.6814 | Val 0.7092/0.5005
EarlyStopping counter: 8/8
Early stopping triggered.
Best Val Acc: 0.5212


In [ ]:
torch.save(model.state_dict(), "best_kobert_model.pt")

import json
with open("label_classes.json", "w", encoding="utf-8") as f:
    json.dump([c for c in le.classes_], f, ensure_ascii=False, indent=2)


NameError: name 'le' is not defined

In [ ]:
train_df['label']

,label
0,19
1,28
2,16
3,7
4,33
...,...
43896,5
43897,19
43898,28
43899,36


In [ ]:
import json
import numpy as np
import pandas as pd

uniq = np.sort(train_df['label'].unique())
assert uniq.min() == 0 and uniq.max() == len(uniq)-1, \
    f"라벨 ID가 연속이 아님: min={uniq.min()}, max={uniq.max()}, n={len(uniq)}"

NUM_LABELS = len(uniq)

name_by_id = {}
for lid, sub in train_df.groupby('label'):
    top_name = (
        sub['interest']
        .astype(str)
        .str.strip()
        .replace('', np.nan)
        .dropna()
        .value_counts()
        .idxmax()
    )
    name_by_id[int(lid)] = top_name

missing = [i for i in range(NUM_LABELS) if i not in name_by_id]
assert not missing, f"이 라벨 ID에 해당하는 interest 이름이 없습니다: {missing}"

label_classes = [name_by_id[i] for i in range(NUM_LABELS)]

with open("label_classes.json", "w", encoding="utf-8") as f:
    json.dump(label_classes, f, ensure_ascii=False, indent=2)

print(f"Saved label_classes.json with {len(label_classes)} classes.")


Saved label_classes.json with 40 classes.


In [ ]:
torch.save(model.state_dict(), "best_kobert_model.pt")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/kobert_artifacts
!cp best_kobert_model.pt /content/drive/MyDrive/kobert_artifacts/
!cp label_classes.json  /content/drive/MyDrive/kobert_artifacts/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!zip -r artifacts.zip best_kobert_model.pt label_classes.json
from google.colab import files
files.download("artifacts.zip")

  adding: best_kobert_model.pt (deflated 7%)
  adding: label_classes.json (deflated 47%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>